# Let's perform some operation on the IRIS dataframe!
In this notebook we will see how to repeat in Pandas the operations we did in the previous labs on numpy

## Exercise 1 - Download and read the IRIS dataset
The Iris dataset we have met it already a number of times.
You can read it directly with pd.read_csv from:
https://archive.ics.uci.edu/ml/machine-learning-databases/iris/iris.data

Remember: the file has no header, so you need to specify it when you read it.
Since it has no header let's assign the following column names to it:
- sepal_length
- sepal_width
- petal_length
- petal_width
- label

In [25]:
import pandas as pd
import numpy as np

pd.set_option("display.width", 400)

In [7]:
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/iris/iris.data"
columns = ["sepal_length", "sepal_width", "petal_length", "petal_width", "label"]

df = pd.read_csv(url, names=columns)
print(df.head())


   sepal_length  sepal_width  petal_length  petal_width        label
0           5.1          3.5           1.4          0.2  Iris-setosa
1           4.9          3.0           1.4          0.2  Iris-setosa
2           4.7          3.2           1.3          0.2  Iris-setosa
3           4.6          3.1           1.5          0.2  Iris-setosa
4           5.0          3.6           1.4          0.2  Iris-setosa


## Exercise 2 - Normalize feature by subtracting means and dividing by standard deviation
For each feature, subtract the mean and divide by the standard deviation.

In [19]:
# Normalize the features: subtract the mean and divide by the standard deviation
features = ["sepal_length", "sepal_width", "petal_length", "petal_width"]

means = df[features].mean()  # one mean per feature
stds = df[features].std()  # one standard deviation per feature

normalized_df = (df[features] - means) / stds  # broadcasting, aligned on the column names
normalized_df["label"] = df["label"]  # keep the label column: Exercise 3 needs it

print(normalized_df.head())

   sepal_length  sepal_width  petal_length  petal_width        label
0     -0.897674     1.028611     -1.336794    -1.308593  Iris-setosa
1     -1.139200    -0.124540     -1.336794    -1.308593  Iris-setosa
2     -1.380727     0.336720     -1.393470    -1.308593  Iris-setosa
3     -1.501490     0.106090     -1.280118    -1.308593  Iris-setosa
4     -1.018437     1.259242     -1.336794    -1.308593  Iris-setosa


## Exercise 3 - Compute the distance of each sample from the mean of its class
- On the normalized features of Exercise 2, compute for each sample the (Euclidean) distance of the sample from its class mean.
- Then create a new column `distance` in the dataframe, that contains the distance.
- Finally, remove the outliers: the rows with distance greater than 2.

The last operation is a simple attempt of outlier detection and it is useful to avoid injecting noise in the data.

In [41]:
df = normalized_df.copy()

print(f"df =\n")
print(df.head())

grouped = df.groupby("label")[features]          # oggetto GroupBy, nessun calcolo ancora
print("\ngrouped =\n")
print(grouped.mean())                            # medie per classe: 3×4

class_means = grouped.transform("mean")          # medie replicate: 150×4
print("\nclass means =\n")
print(class_means.head())

# Euclidean distance of each sample from the mean of its class
df["distance"] = np.sqrt(((df[features] - class_means) ** 2).sum(axis=1))   # adding distance feature
print("\nDistance = \n")
print(df["distance"].head())

# Remove the outliers: the rows farther than 2 from the mean of their class
threshold = 2
filtered_df = df.loc[df["distance"] <= threshold]
print(f"{len(df) - len(filtered_df)} outliers removed, {len(filtered_df)} rows kept")
print(filtered_df.head())

df =

   sepal_length  sepal_width  petal_length  petal_width        label
0     -0.897674     1.028611     -1.336794    -1.308593  Iris-setosa
1     -1.139200    -0.124540     -1.336794    -1.308593  Iris-setosa
2     -1.380727     0.336720     -1.393470    -1.308593  Iris-setosa
3     -1.501490     0.106090     -1.280118    -1.308593  Iris-setosa
4     -1.018437     1.259242     -1.336794    -1.308593  Iris-setosa

grouped =

                 sepal_length  sepal_width  petal_length  petal_width
label                                                                
Iris-setosa         -1.011191     0.839494     -1.300521    -1.250938
Iris-versicolor      0.111907    -0.654990      0.284135     0.166850
Iris-virginica       0.899284    -0.184504      1.016387     1.084088

class means =

   sepal_length  sepal_width  petal_length  petal_width
0     -1.011191     0.839494     -1.300521    -1.250938
1     -1.011191     0.839494     -1.300521    -1.250938
2     -1.011191     0.839494     -